# Padrões Avançados de Recursão no UnifyWeaver

Este caderno demonstra os quatro principais padrões de recursão que o UnifyWeaver pode detectar e otimizar:

1. **Recursão em Cauda (Tail Recursion)** - Laços iterativos com acumuladores
2. **Recursão Linear (Linear Recursion)** - Chamada recursiva única com memoização
3. **Recursão em Árvore (Tree Recursion)** - Múltiplas chamadas recursivas sobre partes da estrutura
4. **Recursão Mútua (Mutual Recursion)** - Predicados que chamam uns aos outros em ciclos

## Objetivos de Aprendizagem

- Compreender diferentes padrões de recursão
- Ver como o UnifyWeaver detecta e otimiza cada padrão
- Comparar características de desempenho
- Aprender quando usar cada padrão

## Configuração

Inicializar o ambiente UnifyWeaver.

In [ ]:
% Load initialization
['../init'].

% Load necessary modules
use_module(unifyweaver(core/recursive_compiler)).
use_module(unifyweaver(core/advanced/pattern_matchers)).

## Padrão 1: Recursão em Cauda (Tail Recursion)

A recursão em cauda utiliza um acumulador para transportar resultados intermediários, e a chamada recursiva é a **última ação** na função.

### Exemplo: Contar Itens em uma Lista

In [ ]:
% Define tail-recursive count_items
:- dynamic count_items/3.

% Base case: empty list, return accumulator
count_items([], Acc, Acc).

% Recursive case: increment accumulator, recurse on tail
count_items([_|T], Acc, N) :-
    Acc1 is Acc + 1,
    count_items(T, Acc1, N).  % ← Tail position!

### Testar em Prolog

In [ ]:
% Test: count items in [a,b,c,d,e]
\+ \+ (
    count_items([a,b,c,d,e], 0, _N),
    format('Count: ~w~n', [_N])
).

### Verificar a Detecção de Padrão

In [ ]:
% Check if detected as tail recursive
\+ \+ (
    is_tail_recursive_accumulator(count_items/3, _AccInfo),
    format('Tail recursive: ~w~n', [_AccInfo])
).

### Compilar para Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(count_items/3, [], _BashCode),
    setup_call_cleanup(
        open('../output/count_items_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled count_items to Bash with tail recursion optimization')
).

### Testar o Bash Gerado

In [ ]:
%%bash
source ../output/count_items_demo.sh
echo "Counting items in [a,b,c,d,e]:"
count_items "[a,b,c,d,e]" 0 ""

## Padrão 2: Recursão Linear (Linear Recursion)

A recursão linear possui **exatamente uma** chamada recursiva por cláusula, com computação ocorrendo após o retorno da chamada recursiva.

### Exemplo: Fatorial (Factorial)

In [ ]:
% Define factorial
:- dynamic factorial/2.

% Base case
factorial(0, 1).

% Recursive case: exactly ONE recursive call
factorial(N, F) :-
    N > 0,
    N1 is N - 1,
    factorial(N1, F1),  % ← One recursive call
    F is N * F1.        % ← Computation after call

### Testar em Prolog

In [ ]:
% Test: factorial of 5
\+ \+ (
    factorial(5, _F),
    format('5! = ~w~n', [_F])
).

### Verificar a Detecção de Padrão

In [ ]:
% Check if detected as linear recursive
is_linear_recursive_streamable(factorial/2),
writeln('✓ Detected as linear recursion').

### Compilar para Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(factorial/2, [], _BashCode),
    % Keep function definitions only; Brush treats sourced scripts as direct execution
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Auto-execute when run directly (not when sourced)"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/factorial_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled factorial to Bash with fold-based linear recursion')
).

### Testar o Bash Gerado

In [ ]:
%%bash
source ../output/factorial_demo.sh
echo "Factorial of 5:"
factorial 5 ""
echo ""
echo "Factorial of 10:"
factorial 10 ""

## Padrão 3: Recursão em Árvore (Tree Recursion)

A recursão em árvore realiza **múltiplas** chamadas recursivas para processar diferentes partes de uma estrutura.

### Exemplo: Soma de Árvore (Tree Sum)

In [ ]:
% Define tree_sum for binary trees
% Tree format: [Value, LeftSubtree, RightSubtree] or []
:- dynamic tree_sum/2.

% Base case: empty tree has sum 0
tree_sum([], 0).

% Recursive case: sum = value + left_sum + right_sum
tree_sum([V, L, R], Sum) :-
    tree_sum(L, LS),   % ← First recursive call
    tree_sum(R, RS),   % ← Second recursive call
    Sum is V + LS + RS.

### Testar em Prolog

In [ ]:
% Test: tree_sum of [5, [3, [1, [], []], []], [2, [], []]]
%       5
%      / \
%     3   2
%    /
%   1
\+ \+ (
    tree_sum([5, [3, [1, [], []], []], [2, [], []]], _Sum),
    format('Tree sum: ~w (expected 11)~n', [_Sum])
).

### Compilar para Bash

In [ ]:
% Compile and save
\+ \+ (
    compile_recursive(tree_sum/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/tree_sum_demo.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('✓ Compiled tree_sum to Bash with tree recursion')
).

### Testar o Bash Gerado

In [ ]:
%%bash
source ../output/tree_sum_demo.sh
echo "Tree sum of [5,[3,[1,[],[]],[]],[2,[],[]]]:"
tree_sum "[5,[3,[1,[],[]],[]],[2,[],[]]]"

## Padrão 4: Recursão Mútua (Mutual Recursion)

A recursão mútua ocorre quando dois ou mais predicados chamam uns aos outros em um ciclo.

### Exemplo: Par (Even) e Ímpar (Odd)

In [ ]:
% Define mutually recursive is_even and is_odd
:- dynamic is_even/1.
:- dynamic is_odd/1.

% is_even base case
is_even(0).

% is_even recursive: N is even if N-1 is odd
is_even(N) :-
    N > 0,
    N1 is N - 1,
    is_odd(N1).  % ← Calls is_odd

% is_odd base case
is_odd(1).

% is_odd recursive: N is odd if N-1 is even
is_odd(N) :-
    N > 1,
    N1 is N - 1,
    is_even(N1).  % ← Calls is_even

### Testar em Prolog

In [ ]:
% Test even/odd
is_even(0), writeln('✓ 0 is even').
is_even(4), writeln('✓ 4 is even').
is_odd(3), writeln('✓ 3 is odd').
is_odd(7), writeln('✓ 7 is odd').

### Verificar Recursão Mútua

In [ ]:
% Build call graph and find SCCs
\+ \+ (
    use_module(unifyweaver(core/advanced/call_graph)),
    use_module(unifyweaver(core/advanced/scc_detection)),

    build_call_graph([is_even/1, is_odd/1], _Graph),
    format('Call graph: ~w~n', [_Graph]),

    find_sccs(_Graph, _SCCs),
    format('SCCs (mutual recursion groups): ~w~n', [_SCCs])
).

### Compilar para Bash

In [ ]:
% Compile the mutual recursion group
\+ \+ (
    use_module(unifyweaver(core/advanced/mutual_recursion)),

    compile_mutual_recursion([is_even/1, is_odd/1], [], _BashCode),
    split_string(_BashCode, "\n", "\r", _BashLines),
    append(_LibraryLines, ["# Main dispatch: route command line calls to functions"|_], _BashLines),
    atomics_to_string(_LibraryLines, "\n", _LibraryCode),
    setup_call_cleanup(
        open('../output/even_odd_demo.sh', write, _Stream),
        write(_Stream, _LibraryCode),
        close(_Stream)),
    writeln('✓ Compiled is_even/is_odd to Bash with shared memoization')
).

### Testar o Bash Gerado

In [ ]:
%%bash
source ../output/even_odd_demo.sh
echo "Testing is_even and is_odd:"
is_even 0 >/dev/null && echo "✓ 0 is even"
is_even 4 >/dev/null && echo "✓ 4 is even"
is_odd 3 >/dev/null && echo "✓ 3 is odd"
is_odd 7 >/dev/null && echo "✓ 7 is odd"
is_even 5 >/dev/null 2>&1 || echo "✓ 5 is not even"

## Comparação de Padrões

Vamos comparar as características de cada padrão:

| Padrão | Chamadas Recursivas | Otimização | Complexidade Espacial | Mais Indicado Para |
|:--------|:----------------|:-------------|:-----------------|:---------|
| **Cauda** | 1 (na posição de cauda) | Laço iterativo | O(1) | Acumuladores, varreduras lineares |
| **Linear** | 1 (em qualquer posição) | Fold + memoização | Tabela memo O(n) | Fibonacci, fatorial |
| **Árvore** | 2+ (partes da estrutura) | Decomposição estrutural | Pilha O(profundidade) | Operações em árvores/grafos |
| **Mútua** | 1+ (entre predicados) | Memoização compartilhada | Tabela compartilhada O(n) | Par/ímpar, definições mútuas |

## Ordem de Detecção de Padrões

O UnifyWeaver tenta casar os padrões na seguinte ordem:

1. **Recursão em Cauda** (mais eficiente)
2. **Recursão Linear** (a menos que proibida)
3. **Recursão em Árvore** (estrutural)
4. **Recursão Mútua** (detecção de Componentes Fortemente Conexos - SCC)
5. **Recursão Básica** (fallback padrão)

Você pode influenciar a detecção com `forbid_linear_recursion/1`.

## Exercício: Sua Vez!

Tente definir e compilar estes predicados:

### 1. Soma com Recursão em Cauda
```prolog
sum_list([], Acc, Acc).
sum_list([H|T], Acc, Sum) :-
    Acc1 is Acc + H,
    sum_list(T, Acc1, Sum).
```

### 2. Fibonacci com Recursão Linear
```prolog
fib(0, 0).
fib(1, 1).
fib(N, F) :-
    N > 1,
    N1 is N - 1,
    N2 is N - 2,
    fib(N1, F1),
    fib(N2, F2),
    F is F1 + F2.
```

### 3. Altura de Árvore
```prolog
tree_height([], 0).
tree_height([_, L, R], H) :-
    tree_height(L, HL),
    tree_height(R, HR),
    H is max(HL, HR) + 1.
```

In [ ]:
% Your code here!


## Resumo

Neste caderno, você aprendeu:

✅ Os quatro principais padrões de recursão no UnifyWeaver

✅ Como definir cada padrão em Prolog

✅ Como o UnifyWeaver detecta e otimiza cada padrão

✅ As características de desempenho de cada padrão

✅ Quando usar cada padrão

## Próximos Passos

Continue para o **Caderno 3: Visualização do Grafo de Chamadas** para aprender sobre análise avançada de código e visualização!